In [2]:
import tensorflow as tf

# Force TensorFlow to share memory politely
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory growth enabled")
    except RuntimeError as e:
        print(e)


In [3]:
# ============================================================
# CELL 1: SETUP & TOKENIZER
# ============================================================
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# Hardcoded from Kaggle - no video dataset needed!
SEQUENCE_LENGTH = 424
MAX_TOKENS = 3000
NUM_FEATURES = 232
CSV_TRAIN = 'M:\Term 10\Grad\SLR Main\how2sign_realigned_train.csv' 

# Load the CSV to build the vocabulary
try:
    train_df = pd.read_csv(CSV_TRAIN, sep='\t', on_bad_lines='skip')
    if 'SENTENCE' not in train_df.columns:
        train_df = pd.read_csv(CSV_TRAIN, on_bad_lines='skip')
    train_sentences = train_df['SENTENCE'].fillna('').tolist()
except FileNotFoundError:
    print(f"❌ Error: '{CSV_TRAIN}' not found. Make sure it is in the same folder as this notebook.")
    train_sentences = []

# Build Tokenizer
tokenizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int'
)

if train_sentences:
    print("Adapting tokenizer...")
    tokenizer.adapt(train_sentences)
    vocab = tokenizer.get_vocabulary()
    print(f"✅ Vocabulary ready! Size: {len(vocab)}")
else:
    vocab = []

# Function to translate model numbers back to English
def decode_indices(indices):
    words = []
    for idx in indices:
        if idx == 0: continue
        idx = idx - 1
        if 0 <= idx < len(vocab):
            w = vocab[idx]
            if w not in ['', '[UNK]']: words.append(w)
    return ' '.join(words)


Adapting tokenizer...
✅ Vocabulary ready! Size: 3000


In [4]:
# ============================================================
# CELL 2: MODEL ARCHITECTURE
# ============================================================
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LSTM, Bidirectional, Input, MultiHeadAttention, LayerNormalization, Softmax

vocab_size_ctc = MAX_TOKENS + 1

# Define the inputs (1 video, 424 frames, 232 features)
input_frames = Input(shape=(SEQUENCE_LENGTH, 232), name='input')

# Encoder
x1 = Bidirectional(LSTM(256, return_sequences=True), name='bilstm_1')(input_frames)
x1 = BatchNormalization(name='bn_1')(x1)

# Attention Mechanism
attn_out = MultiHeadAttention(num_heads=4, key_dim=64, name='mha')(x1, x1)
x = LayerNormalization(name='ln_1')(x1 + attn_out)
x = Dropout(0.3, name='drop_1')(x)

x = Bidirectional(LSTM(256, return_sequences=True), name='bilstm_2')(x)
x = BatchNormalization(name='bn_2')(x)
x = Dropout(0.3, name='drop_2')(x)

# Output Layer
logits = Dense(vocab_size_ctc, name='logits')(x)
y_pred = Softmax(name='prediction')(logits)

# Build the final inference model
model_inference = tf.keras.models.Model(inputs=input_frames, outputs=y_pred, name='ctc_inference')
print("✅ Model architecture successfully built in memory!")


✅ Model architecture successfully built in memory!


In [5]:
# ============================================================
# CELL 3: LOAD WEIGHTS & SAFE WEBCAM INFERENCE
# ============================================================
import os

# ── 1. HARDWARE FIX (CRITICAL FOR WINDOWS) ──────────────────
# This forces TensorFlow to use the CPU, completely bypassing 
# the CUDA/BLAS kernel crash you were experiencing.
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import time
from collections import deque

print("System Check: Forcing CPU mode to prevent BLAS crashes... ✅")

# ── 2. FILE CONFIRMATION TEST ───────────────────────────────
# We use the exact filename from your screenshot
WEIGHTS_PATH = 'best_cslr_model.weights (1).h5'

print(f"System Check: Looking for '{WEIGHTS_PATH}' in current directory...")
if not os.path.exists(WEIGHTS_PATH):
    print(f"❌ CRITICAL ERROR: File not found!")
    raise FileNotFoundError(f"Missing weights file: {WEIGHTS_PATH}")
print("System Check: File located! ✅")

# ── 3. WEIGHT LOADING TEST ──────────────────────────────────
print("System Check: Loading neural network weights...")
try:
    model_inference.load_weights(WEIGHTS_PATH)
    print("System Check: Weights loaded successfully! ✅")
except Exception as e:
    print(f"❌ CRITICAL ERROR: Failed to load weights: {e}")
    raise e

# ── 4. STABILIZATION TRACKER ────────────────────────────────
class StabilizationTracker:
    def __init__(self, window_size=15, majority_ratio=0.6, cooldown_s=1.0):
        self.window_size = window_size
        self.majority_ratio = majority_ratio
        self.cooldown_s = cooldown_s
        self.buffer = deque(maxlen=window_size)
        self.last_commit_time = 0
        self.last_committed_word = ""
        self.sentence = []

    def update(self, predicted_word):
        if not predicted_word: return None
        self.buffer.append(predicted_word)
        if len(self.buffer) < self.window_size: return None
        
        counts = {}
        for w in self.buffer: counts[w] = counts.get(w, 0) + 1
        
        top_word = max(counts, key=counts.get)
        top_ratio = counts[top_word] / self.window_size
        
        now = time.time()
        if top_ratio >= self.majority_ratio:
            if top_word != self.last_committed_word and (now - self.last_commit_time) > self.cooldown_s:
                self.sentence.append(top_word)
                self.last_committed_word = top_word
                self.last_commit_time = now
                self.buffer.clear()
                return top_word
        return None

# ── 5. MEDIAPIPE CONFIGURATION ──────────────────────────────
mp_holistic = mp.solutions.holistic
_FACE_KP_INDICES = (
    [17,18,19,20,21] + [22,23,24,25,26] + [36,37,38,39,40,41] + [42,43,44,45,46,47] + 
    [68,69] + [27,28,29,30] + [33] + [48,49,50,51,52,53,54,55,56,57,58,59] + 
    [60,61,62,63,64,65,66,67]
)  
_MP_FACE_TO_OP70 = {
    17:70, 18:63, 19:105, 20:66, 21:107, 22:336, 23:296, 24:334, 25:293, 26:300, 
    36:33, 37:160, 38:158, 39:133, 40:153, 41:144, 42:362, 43:385, 44:387, 45:263, 
    46:373, 47:380, 68:468, 69:473, 27:6, 28:197, 29:195, 30:5, 33:1, 48:61, 49:185, 
    50:40, 51:39, 52:37, 53:0, 54:267, 55:269, 56:270, 57:409, 58:291, 59:375, 
    60:78, 61:191, 62:80, 63:81, 64:311, 65:310, 66:415, 67:308                   
}

def mediapipe_to_232dim(results):
    pose = np.zeros((25, 2), dtype=np.float32)
    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def set_p(op_idx, mp_idx):
            if mp_idx < len(lm): pose[op_idx] = [lm[mp_idx].x, lm[mp_idx].y]
        set_p(0, 0); set_p(2, 12); set_p(3, 14); set_p(4, 16); set_p(5, 11); set_p(6, 13)
        set_p(7, 15); set_p(9, 24); set_p(10, 26); set_p(11, 28); set_p(12, 23); set_p(13, 25)
        set_p(14, 27); set_p(15, 5); set_p(16, 2); set_p(17, 8); set_p(18, 7); set_p(19, 31)
        set_p(21, 29); set_p(22, 32); set_p(24, 30)
        if pose[2].any() and pose[5].any(): pose[1] = (pose[2] + pose[5]) / 2.0
        if pose[9].any() and pose[12].any(): pose[8] = (pose[9] + pose[12]) / 2.0

    left_hand = np.zeros((21, 2), dtype=np.float32)
    if results.left_hand_landmarks:
        for k, pt in enumerate(results.left_hand_landmarks.landmark): left_hand[k] = [pt.x, pt.y]

    right_hand = np.zeros((21, 2), dtype=np.float32)
    if results.right_hand_landmarks:
        for k, pt in enumerate(results.right_hand_landmarks.landmark): right_hand[k] = [pt.x, pt.y]

    face = np.zeros((49, 2), dtype=np.float32)
    if results.face_landmarks:
        mesh = results.face_landmarks.landmark
        for k, op_idx in enumerate(_FACE_KP_INDICES):
            mp_idx = _MP_FACE_TO_OP70.get(op_idx)
            if mp_idx is not None and mp_idx < len(mesh): face[k] = [mesh[mp_idx].x, mesh[mp_idx].y]

    return np.concatenate([pose.flatten(), left_hand.flatten(), right_hand.flatten(), face.flatten()])

# ── 6. WEBCAM LOOP ──────────────────────────────────────────
print("\n" + "="*40)
print("🎥 ALL SYSTEMS GO. STARTING WEBCAM...")
print("   (Press 'q' in the video window to stop)")
print("="*40 + "\n")

cap = cv2.VideoCapture(0)
tracker = StabilizationTracker(window_size=15, majority_ratio=0.6)
frame_buffer = deque(maxlen=SEQUENCE_LENGTH)

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: 
            print("❌ Error: Could not grab frame from camera.")
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        features = mediapipe_to_232dim(results)
        frame_buffer.append(features)

        # Run prediction every 5 frames
        if len(frame_buffer) > 10 and frame_count % 5 == 0:
            X = np.expand_dims(np.stack(frame_buffer), axis=0)
            
            # Pad sequence if needed
            if X.shape[1] < SEQUENCE_LENGTH:
                pad = np.zeros((1, SEQUENCE_LENGTH - X.shape[1], 232))
                X = np.concatenate([X, pad], axis=1)

            # Predict (using CPU)
            preds = model_inference.predict(X, verbose=0)
            input_lengths = np.array([min(len(frame_buffer), SEQUENCE_LENGTH)])
            decoded, _ = tf.keras.backend.ctc_decode(preds, input_length=input_lengths, greedy=True)
            
            sentence = decode_indices(decoded[0][0].numpy())
            tracker.update(sentence)

        # UI Overlay
        final_text = ' '.join(tracker.sentence) if tracker.sentence else 'Listening...'
        cv2.rectangle(image, (0, 0), (640, 80), (0, 0, 0), -1)
        cv2.putText(image, final_text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow('Sign Language Translation (Press Q to quit)', image)

        frame_count += 1
        if cv2.waitKey(10) & 0xFF == ord('q'): 
            break

cap.release()
cv2.destroyAllWindows()
print("🛑 Webcam closed successfully.")


System Check: Forcing CPU mode to prevent BLAS crashes... ✅
System Check: Looking for 'best_cslr_model.weights (1).h5' in current directory...
System Check: File located! ✅
System Check: Loading neural network weights...
System Check: Weights loaded successfully! ✅

🎥 ALL SYSTEMS GO. STARTING WEBCAM...
   (Press 'q' in the video window to stop)

🛑 Webcam closed successfully.
